In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ---------------------------------------------------------------------------
# GLOBALS — trimmed to match real economic_data.feather coverage (FIX #20)
# economic_data.feather only has usable daily density from 2013-03-01
# through 2024-01-26 (checked directly against the file): Jan-Feb 2013 is
# too sparse to trust (1 day of coverage in all of Jan 2013), and nothing
# exists past 2024-01-26. Rather than fabricate calendar data for the gap,
# price + GDELT are now trimmed to the SAME window so every feature source
# has genuine coverage across the full dataset -- no more silent
# fillna(0)-as-"no macro event" for a period where the real answer is
# "we simply have no data."
# ---------------------------------------------------------------------------
START_DATE = "2013-03-01"
END_DATE   = "2024-01-26"

In [ ]:
"""
===========================================================================
 FX MULTI-PAIR FEATURE PIPELINE -- v4
===========================================================================
Price -> Returns -> Correlation -> Johansen -> VECM -> GARCH -> DCC
      -> Macro calendar -> GDELT -> Regimes -> Rolling -> Feature select -> CSV

Output: ~82 columns (93 if the newer GDELT export is used), against 616 before.
        25 price | 21 calendar | 26 GDELT | 10 id/target

---------------------------------------------------------------------------
A. ROOT CAUSE OF THE 616-COLUMN DATASET
---------------------------------------------------------------------------
Per-group frames were named after the PAIRS they held, so every group had
different column names. Each pd.concat() therefore produced the UNION of all
groups columns, and the ffill()/fillna(0) that followed filled the gaps with
values belonging to OTHER groups -- ~575 of 616 columns were a different
group value frozen at a block boundary. A global "dead column" check saw
nothing wrong because the fabricated values varied ACROSS groups.

FIX: columns are named by SLOT (1/2/3), sorted pair order, with the pair
identities kept in pair_1/pair_2/pair_3. Every group emits one identical
schema, so the concat has no gaps and no fill is possible.
Check it with:  df.groupby("group_id").nunique() <= 1

---------------------------------------------------------------------------
B. LEAKAGE FIXED
---------------------------------------------------------------------------
- macro_surprise_*_lead1 REMOVED. A scheduled release is known in advance;
  its SURPRISE (actual - forecast) does not exist until publication. The old
  roll_cols filter only excluded names containing an event-type keyword, so
  every surprise column got a shift(-1).
- rho _change now uses a GROUPED diff (was measuring the first row of a group
  against the last row of the previous group).
- panel ffill/bfill now inside groupby("group_id").
- event_flag no longer back-fills a future threshold.
- STILL NOT LEAK-FREE: `regime` (HMM) is fit once over the full history.
  Exclude it from hist_exog for an honest split, or refit per fold.

---------------------------------------------------------------------------
C. PRICE WAS MISSING ENTIRELY
---------------------------------------------------------------------------
`returns` was computed in main(), used to rank pairs and fit GARCH, then
discarded. The dataset held only SECOND-ORDER price information -- volatility,
correlation, spread -- and never the actual price move, while carrying 63
GDELT columns. Added per slot: ret_k, ret_k_lag1, ret_k_mean_5, ret_k_std_20,
plus zscore_lag1 / zscore_diff_1 / zscore_diff_5 and dow.

---------------------------------------------------------------------------
D. CALENDAR MADE MEANINGFUL
---------------------------------------------------------------------------
importance {-1:1, 0:2, 1:3} with a daily SUM meant six filler releases
outweighed one CPI print -- and economic_data.feather holds 130,565 rows at
importance -1 against 4,667 at +1, so macro_k measured event COUNT, not
importance. Reweighted to {-1:0.25, 0:1.0, 1:4.0}.
Also: _3d now rolls over a continuous daily calendar (it used to roll over the
pivot own rows, so "3 days" could span a week), and timestamps are normalized
in local time instead of shifting through UTC.
KEPT: macro_k, macro_surprise_k, rate / inflation / labor types, lead1.
DROPPED: growth / pmi / trade types, the _3d windows.

---------------------------------------------------------------------------
E. GDELT: WHAT IT ACTUALLY MEASURES, AND THE SCALE PROBLEM
---------------------------------------------------------------------------
The BigQuery export filters GoldsteinScale <= -5, i.e. CONFLICTUAL events
only. avg_tone is therefore the tone of conflict coverage, not general news
sentiment. Treat the block as a GEOPOLITICAL STRESS measure.

mentions_diff for AUDUSD=X reads -16766, -15887, -14853 every single day --
the US out-publishing Australia ~10x, not information. And GDELT own sourcing
expanded sharply in 2015: mean per-row mentions went 4,429 (2010) -> 23,069
(2015) -> 12,168 (2024), a 5x level break inside the training window.
FIX: counts become a relative share (base-quote)/(base+quote), bounded
[-1,1], so the country-size effect and the coverage break divide out.

KEPT: avg_tone_diff, avg_goldstein_diff (corr with tone is 0.004 -- narrow but
INDEPENDENT: how severe the severe events were), shock_rel, mentions_rel.
DROPPED: articles_diff (corr with mentions_diff = 1.000 exactly), the 3- and
14-day windows, tone_mentions_interaction.
OPTIONAL: conflict_share_diff and tone_all_diff are built automatically IF the
newer BigQuery export is present -- the one that keeps total_event_count /
total_mentions / avg_tone_all alongside the filtered figures, via conditional
aggregation instead of a WHERE clause. Same bytes scanned, so same cost.

---------------------------------------------------------------------------
F. TRAINING SETUP
---------------------------------------------------------------------------
   unique_id   = group_id
   ds          = Date
   y           = df.groupby("group_id")["zscore"].shift(-1) - df["zscore"]
   static_exog = ["pair_1", "pair_2", "pair_3"]
   futr_exog   = ["macro_1_lead1", "macro_2_lead1", "macro_3_lead1", "dow"]
   hist_exog   = everything else, minus Date/group/group_id/time_idx/spread
                 and minus `regime` (see B)

SPLIT BY DATE, not by row -- the 20 groups share pairs with each other, so a
random split leaks. Suggested: train <= 2021-12, val 2022, test 2023+.
Beat the predict-zero and naive baselines printed by the check cell before
believing any model result.
"""


In [ ]:

"""

/1
=================================================
FX Multi-Pair Feature Dataset Pipeline (cleaned)
=================================================

Rebuilds: Price -> Returns -> Correlation -> Cointegration -> VECM -> GARCH
-> DCC -> Macro Calendar -> GDELT -> Regimes -> Rolling Features -> Final CSV

Fixes applied vs. the original exploratory notebook
-----------------------------------------------------
1. LEAKAGE: GARCH input scaling, HMM regime fitting, and the vol/macro
   percentile-regime cutoffs all used to be computed on the *whole* series
   (train+test). They now use EXPANDING windows, so a value at time t only
   ever depends on data up to t.
2. LEAKAGE: `event_lead_1/2` (future GDELT shock flags) removed entirely --
   GDELT news shocks are not knowable in advance, unlike scheduled macro
   releases. Only macro-calendar `_lead1` columns are kept as "known future"
   features, since those really are known ahead of time (economic calendar).
3. BLOAT: macro calendar features are now aggregated per *group* (only the
   pairs actually inside that triplet), instead of broadcasting all 27
   pairs' macro columns onto every group row.
4. Group-id dedup is centralized in `build_group_sets`, run once right after
   group construction, instead of being patched on after the fact.
5. Single CSV export at the very end. No duplicate/duplicate-named exports.
6. Deprecated pandas syntax replaced (`.ffill()/.bfill()` instead of
   `fillna(method=...)`), dead/unused code removed.

Usage
-----
Run top to bottom, or import and call `main()`. You need two local files
before running the calendar/GDELT stages:
    economic_data.feather   (or .csv with the same columns)
    gdelt.csv
"""

!pip install arch hmmlearn yfinance

# Global variables
PAIRS = [
    "EURUSD=X", "GBPUSD=X", "USDJPY=X", "AUDUSD=X", "USDCAD=X", "USDCHF=X", "NZDUSD=X",
    "EURGBP=X", "EURJPY=X", "EURAUD=X", "EURCAD=X", "EURCHF=X", "EURNZD=X",
    "GBPJPY=X", "GBPAUD=X", "GBPCAD=X", "GBPCHF=X", "GBPNZD=X",
    "AUDJPY=X", "AUDNZD=X", "AUDCAD=X", "AUDCHF=X",
    "CADJPY=X", "CHFJPY=X", "NZDJPY=X", "NZDCAD=X", "NZDCHF=X",
]


import re
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 15.8 MB/s eta 0:00:00


In [ ]:

# ---------------------------------------------------------------------------
# 1. PRICE DATA
# ---------------------------------------------------------------------------
def download_prices(pairs=PAIRS, start=START_DATE, end=END_DATE) -> pd.DataFrame:
    import yfinance as yf

    data = yf.download(pairs, start=start, end=end)["Close"]
    data = data.ffill().bfill()
    return data


In [ ]:

# ---------------------------------------------------------------------------
# 2. ROLLING CORRELATION -> CANDIDATE PAIR GROUPS
# ---------------------------------------------------------------------------
def top_correlated_pairs(
    returns: pd.DataFrame, window: int = 20, top_n: int = 2, threshold: float = 0.8
) -> dict:
    """
    For each base pair, rank the other pairs by (frequency of being
    strongly correlated, average correlation, max streak length) using a
    rolling window, and keep the top_n.
    """
    freq = {c: {} for c in returns.columns}
    corr_sum = {c: {} for c in returns.columns}
    streak = {c: {} for c in returns.columns}
    cur_streak = {c: {} for c in returns.columns}

    for date in returns.index[window:]:
        window_data = returns.loc[returns.index <= date].iloc[-window:]
        corr_matrix = window_data.corr()

        for base in corr_matrix.columns:
            for other in corr_matrix.columns:
                if other == base:
                    continue
                c = corr_matrix.loc[base, other]
                if pd.isna(c):
                    continue
                corr_sum[base][other] = corr_sum[base].get(other, 0.0) + c
                if abs(c) >= threshold:
                    freq[base][other] = freq[base].get(other, 0) + 1
                    cur_streak[base][other] = cur_streak[base].get(other, 0) + 1
                    streak[base][other] = max(
                        streak[base].get(other, 0), cur_streak[base][other]
                    )
                else:
                    cur_streak[base][other] = 0

    final_top_pairs = {}
    n_windows = max(len(returns.index) - window, 1)
    for base in returns.columns:
        candidates = []
        for other, f in freq[base].items():
            avg_corr = corr_sum[base][other] / n_windows
            candidates.append((other, f, avg_corr, streak[base].get(other, 0)))
        candidates.sort(key=lambda x: (x[1], abs(x[2]), x[3]), reverse=True)
        final_top_pairs[base] = candidates[:top_n]

    return final_top_pairs


def build_corr_dict(final_top_pairs: dict, top_n: int = 2) -> dict:
    return {base: [p[0] for p in lst[:top_n]] for base, lst in final_top_pairs.items()}



In [ ]:
# ---------------------------------------------------------------------------
# 3. STATIONARITY (ADF)
# ---------------------------------------------------------------------------
def adf_summary(data: pd.DataFrame) -> pd.DataFrame:
    diff_return = data.diff().dropna()
    log_return = np.log(data).diff().dropna()

    rows = []
    for name, series_set, label in [
        ("level", data, "level"),
        ("diff", diff_return, "diff"),
        ("log_return", log_return, "log_return"),
    ]:
        for col in series_set.columns:
            stat, pvalue = adfuller(series_set[col].dropna())[:2]
            rows.append(
                {"pair": col, "transform": label, "stat": stat,
                 "pvalue": pvalue, "stationary": pvalue < 0.05}
            )
    return pd.DataFrame(rows)



In [ ]:
# ---------------------------------------------------------------------------
# 4. JOHANSEN COINTEGRATION -> 3-PAIR GROUPS
# ---------------------------------------------------------------------------
def johansen_confidence_scores(group_df: pd.DataFrame, det_order=0, k_ar_diff=1):
    group_df = group_df.dropna()
    result = coint_johansen(group_df, det_order, k_ar_diff)

    scores_90, scores_95 = [], []
    for i in range(len(result.lr1)):
        trace_stat = result.lr1[i]
        cv_90, cv_95 = result.cvt[i, 0], result.cvt[i, 1]
        scores_90.append(trace_stat / cv_90)
        scores_95.append(trace_stat / cv_95)
    return scores_90, scores_95


def build_group_sets(corr_dict: dict, data: pd.DataFrame, threshold: float = 0.95):
    """
    Build 3-pair cointegrated groups from the correlation dictionary, then
    dedup on the *set* of pairs (order-independent) so the same triplet
    reached from two different base pairs only appears once.
    """
    groups_90, groups_95 = [], []
    for base, correlated in corr_dict.items():
        group = [base] + correlated
        group_df = data[group].dropna()
        if group_df.shape[0] < 10:
            continue

        scores_90, scores_95 = johansen_confidence_scores(group_df)
        if max(scores_90) >= threshold:
            groups_90.append(group)
        if max(scores_95) >= threshold:
            groups_95.append(group)

    def dedup(groups):
        seen, out = set(), []
        for g in groups:
            key = frozenset(g)
            if key not in seen:
                seen.add(key)
                out.append(sorted(g))
        return out

    return dedup(groups_90), dedup(groups_95)



In [ ]:
# ---------------------------------------------------------------------------
# 5. VECM -> RESIDUALS + SPREAD
# ---------------------------------------------------------------------------
def fit_vecm_for_groups(data, groups, diff_lags=1, deterministic="co",
                         min_rows=30, rank=1):
    vecm_models, vecm_residuals, spread_dict, zscore_dict = {}, {}, {}, {}

    for group in groups:
        group_key = tuple(group)
        group_df = data[group].dropna()
        if group_df.shape[0] < min_rows:
            continue

        model = VECM(group_df, k_ar_diff=diff_lags, coint_rank=rank,
                      deterministic=deterministic)
        res = model.fit()

        vecm_models[group_key] = res
        resid = pd.DataFrame(res.resid, index=group_df.index[diff_lags + 1:],
                              columns=group)
        vecm_residuals[group_key] = resid

        beta = res.beta[:, 0]
        spread = group_df.values @ beta
        spread = pd.Series(spread, index=group_df.index, name="spread")
        spread_dict[group_key] = spread
        zscore_dict[group_key] = (spread - spread.expanding().mean()) / spread.expanding().std()

    return vecm_models, vecm_residuals, spread_dict, zscore_dict




In [ ]:
# ---------------------------------------------------------------------------
# 6. GARCH (expanding-window standardization -- fixes leakage)
# ---------------------------------------------------------------------------
def _expanding_standardize(series: pd.Series) -> pd.Series:
    """
    Standardize using only information available up to time t (expanding
    std), instead of the whole-sample std used in the original notebook.
    """
    exp_std = series.expanding(min_periods=20).std()
    exp_std = exp_std.bfill()
    return series / exp_std.replace(0, np.nan).ffill().bfill()


def fit_garch_on_residuals(vecm_residuals: dict):
    garch_models, garch_volatility, garch_std_resid = {}, {}, {}

    for group, resid_df in vecm_residuals.items():
        garch_models[group] = {}
        garch_volatility[group] = pd.DataFrame(index=resid_df.index)
        garch_std_resid[group] = pd.DataFrame(index=resid_df.index)

        for col in resid_df.columns:
            series = resid_df[col].dropna()
            if series.std() == 0:
                continue
            series = _expanding_standardize(series)

            model = arch_model(series, vol="Garch", p=1, q=1, dist="t")
            res = model.fit(disp="off")

            garch_models[group][col] = res
            garch_volatility[group][col] = res.conditional_volatility
            garch_std_resid[group][col] = res.std_resid

    return garch_models, garch_volatility, garch_std_resid


def fit_garch_on_returns(returns: pd.DataFrame):
    returns_volatility = {}
    for col in returns.columns:
        series = returns[col].dropna()
        if series.std() == 0:
            continue
        series = _expanding_standardize(series)
        model = arch_model(series, vol="Garch", p=1, q=1, dist="t")
        res = model.fit(disp="off")
        returns_volatility[col] = res.conditional_volatility
    return returns_volatility


def fit_garch_on_spread(spread_dict: dict):
    spread_volatility = {}
    for group, spread in spread_dict.items():
        series = spread.dropna()
        if series.std() == 0:
            continue
        series = _expanding_standardize(series)
        model = arch_model(series, vol="Garch", p=1, q=1, dist="t")
        res = model.fit(disp="off")
        spread_volatility[group] = res.conditional_volatility
    return spread_volatility




In [ ]:
# ---------------------------------------------------------------------------
# 7. DCC (dynamic conditional correlation)
# ---------------------------------------------------------------------------
class DCC_local:
    """Engle (2002) DCC(1,1), fit with fixed smoothing parameters."""

    def __init__(self, a=0.01, b=0.98):
        self.a, self.b = a, b
        self.Rt = None

    def fit(self, eps: np.ndarray) -> np.ndarray:
        T, N = eps.shape
        Qbar = np.cov(eps.T)
        Qt = Qbar.copy()
        Rt = np.zeros((T, N, N))
        for t in range(T):
            ut = np.outer(eps[t], eps[t])
            Qt = (1 - self.a - self.b) * Qbar + self.a * ut + self.b * Qt
            diag = np.sqrt(np.diag(Qt))
            Rt[t] = Qt / np.outer(diag, diag)
        self.Rt = Rt
        return Rt


# WHY (root cause of the 616-column dataset):
#   The old version named each group's correlation columns after the PAIRS
#   in that group (rho_AUDCAD_AUDUSD, rho_EURUSD_USDCHF, ...). Because every
#   group holds different pairs, every group's DataFrame had DIFFERENT column
#   names. pd.concat() on frames with different columns silently produces the
#   UNION of all columns with NaN in the gaps -- and the .ffill().bfill() that
#   followed then filled those gaps with numbers carried in from OTHER GROUPS.
#   Result: 46 correlation columns on every row, of which 3 were real and 43
#   were another group's value frozen at a block boundary. The .diff() was
#   also ungrouped, so it returned 0 for the frozen columns (the 95%-zeros
#   pattern) and a meaningless jump at each group boundary.
#
# NEW APPROACH — SLOT NAMING:
#   Sort the 3 pairs in a group and name the columns after the SLOT, not the
#   pair: rho_12, rho_13, rho_23 / sigma_1..3 / macro_1..3. The actual pair
#   names are kept in pair_1/pair_2/pair_3 so nothing is lost. Every group now
#   produces the IDENTICAL set of column names, which makes concat safe by
#   construction -- there are no gaps left to fill, so no fabricated values
#   are possible. It also means the model learns one general rule ("how does
#   slot-1 volatility relate to the 1-2 correlation") from all 20 groups at
#   once, instead of a separate rule per group.

SLOT_PAIRS = [(1, 2), (1, 3), (2, 3)]


def build_dcc_dataset(garch_std_resid: dict) -> pd.DataFrame:
    all_features = []

    for group, std_resid_df in garch_std_resid.items():
        srd = std_resid_df.dropna()
        if srd.empty:
            continue

        names = list(srd.columns)
        if len(names) != 3:
            # slot schema assumes triplets; anything else is skipped loudly
            print(f"SKIP {group}: expected 3 pairs, got {len(names)}")
            continue

        # NEW: canonical slot order == sorted pair names, so "slot 1" means
        # the same thing in every group and matches normalize_group_id().
        order = sorted(range(3), key=lambda i: names[i])
        srd = srd.iloc[:, order]
        names = [names[i] for i in order]

        eps = srd.values
        if np.any(eps.std(axis=0) == 0):
            continue

        Rt = DCC_local().fit(eps)

        feats = pd.DataFrame(index=srd.index)
        for a, b in SLOT_PAIRS:
            feats[f"rho_{a}{b}"] = Rt[:, a - 1, b - 1]       # NEW: slot names
        feats["group"] = "_".join(group)
        feats["group_id"] = "_".join(names)                   # already sorted
        for k, n in enumerate(names, start=1):
            feats[f"pair_{k}"] = n                            # NEW: keeps identity

        all_features.append(feats.rename_axis("Date").reset_index())

    final_dcc_df = pd.concat(all_features, ignore_index=True)
    # OLD: final_dcc_df = final_dcc_df.ffill().bfill()
    # NEW: nothing to fill -- every group contributes the same columns.
    final_dcc_df["Date"] = pd.to_datetime(final_dcc_df["Date"])
    final_dcc_df = final_dcc_df.sort_values(["group_id", "Date"]).reset_index(drop=True)

    # OLD: new_features[f"{col}_change"] = final_dcc_df[col].diff()   <- ungrouped
    # NEW: diff within each group, so the first row of a group is not measured
    #      against the last row of the previous group.
    new_features = {}
    for a, b in SLOT_PAIRS:
        col = f"rho_{a}{b}"
        new_features[f"{col}_change"] = final_dcc_df.groupby("group_id")[col].diff().fillna(0.0)
        new_features[f"{col}_abs"] = final_dcc_df[col].abs()
    final_dcc_df = pd.concat(
        [final_dcc_df, pd.DataFrame(new_features, index=final_dcc_df.index)], axis=1
    ).copy()

    return final_dcc_df


In [ ]:
# ---------------------------------------------------------------------------
# 8. MERGE DCC + SIGMA + RETURNS + SPREAD/ZSCORE, BUILD time_idx, DEDUP
# ---------------------------------------------------------------------------
def normalize_group_id(group_str: str) -> str:
    pairs = re.findall(r"[A-Z]{6}=X", group_str)
    return "_".join(sorted(pairs))


def build_panel(final_dcc_df: pd.DataFrame, returns_volatility: dict,
                spread_dict: dict, zscore_dict: dict,
                returns: pd.DataFrame = None) -> pd.DataFrame:
    """
    WHY (three fixes):

    1. SIGMA BROADCAST.
       OLD: merged sigma on Date alone, so all 27 pairs' volatilities landed
            on every row. NEW: joined on (Date, pair_k) -> sigma_1/2/3 only.

    2. UNGROUPED FILL.
       OLD: panel.ffill().bfill() on the whole frame, so the first rows of a
            group inherited the last rows of the previous group.
       NEW: fill happens inside groupby("group_id").

    3. PRICE WAS MISSING ENTIRELY  <-- NEW in v3.
       `returns` was computed in main(), used to rank pairs and fit GARCH,
       and then discarded. The exported dataset therefore contained only
       SECOND-ORDER price information (volatility, correlation, spread) and
       never the actual price move. Lagged returns of the three legs are
       among the most obvious predictors of tomorrow's spread change, so
       they are now joined per slot, plus short-horizon momentum/dispersion
       and the z-score's own recent history.
    """
    df = final_dcc_df.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    # --- sigma per slot -------------------------------------------------
    sigma_df = pd.concat(returns_volatility, axis=1)
    sigma_df.index = pd.to_datetime(sigma_df.index)
    sigma_long = (sigma_df.rename_axis("Date").reset_index()
                          .melt(id_vars="Date", var_name="pair", value_name="sigma"))
    for k in (1, 2, 3):
        df = df.merge(sigma_long.rename(columns={"pair": f"pair_{k}", "sigma": f"sigma_{k}"}),
                      on=["Date", f"pair_{k}"], how="left")

    # --- NEW: raw returns per slot --------------------------------------
    if returns is not None:
        ret_long = (returns.rename_axis("Date").reset_index()
                           .melt(id_vars="Date", var_name="pair", value_name="ret"))
        ret_long["Date"] = pd.to_datetime(ret_long["Date"])
        for k in (1, 2, 3):
            df = df.merge(ret_long.rename(columns={"pair": f"pair_{k}", "ret": f"ret_{k}"}),
                          on=["Date", f"pair_{k}"], how="left")

    # --- spread + zscore ------------------------------------------------
    spread_rows = []
    for group_key, spread in spread_dict.items():
        gid = normalize_group_id("_".join(group_key))
        zscore = zscore_dict[group_key]
        spread_rows.append(pd.DataFrame({
            "Date": pd.to_datetime(spread.index),
            "spread": spread.values,
            "zscore": zscore.values,
            "group_id": gid,
        }))
    spread_df = (pd.concat(spread_rows, ignore_index=True)
                   .drop_duplicates(subset=["group_id", "Date"], keep="first"))
    panel = df.merge(spread_df, on=["group_id", "Date"], how="left")

    panel = panel.drop_duplicates(subset=["Date", "group_id"], keep="first")
    panel = panel.sort_values(["group_id", "Date"]).reset_index(drop=True)
    panel["time_idx"] = panel.groupby("group_id").cumcount()

    ID_COLS = ["Date", "group", "group_id", "pair_1", "pair_2", "pair_3"]
    fill_cols = [c for c in panel.columns if c not in ID_COLS]
    panel[fill_cols] = panel.groupby("group_id")[fill_cols].ffill()
    panel[fill_cols] = panel.groupby("group_id")[fill_cols].bfill()

    # --- NEW: price history features, all computed INSIDE the group ------
    if returns is not None:
        for k in (1, 2, 3):
            g = panel.groupby("group_id")[f"ret_{k}"]
            panel[f"ret_{k}_lag1"]   = g.shift(1)
            panel[f"ret_{k}_mean_5"] = g.transform(lambda x: x.rolling(5,  min_periods=1).mean())
            panel[f"ret_{k}_std_20"] = g.transform(lambda x: x.rolling(20, min_periods=2).std())

    # weekday: known in advance, so it belongs in futr_exog. FX liquidity and
    # the release calendar both have strong day-of-week structure.
    panel["dow"] = panel["Date"].dt.dayofweek

    gz = panel.groupby("group_id")["zscore"]
    panel["zscore_lag1"]   = gz.shift(1)
    panel["zscore_diff_1"] = gz.diff(1)
    panel["zscore_diff_5"] = gz.diff(5)

    num_cols = [c for c in panel.columns if c not in ID_COLS]
    panel[num_cols] = panel[num_cols].fillna(0)
    return panel


In [ ]:
# ---------------------------------------------------------------------------
# 9. MACRO CALENDAR (per group, per slot)
# ---------------------------------------------------------------------------
COUNTRY_TO_CURRENCY = {
    "US": "USD", "EU": "EUR", "GB": "GBP", "JP": "JPY",
    "AU": "AUD", "NZ": "NZD", "CA": "CAD", "CH": "CHF",
}

HIGH_IMPACT_TYPES = ["inflation", "growth", "labor", "rate", "trade", "pmi"]


def categorize_event(indicator: str) -> str:
    ind = (indicator or "").lower()
    rules = [
        (("cpi", "inflation"), "inflation"),
        (("gdp",), "growth"),
        (("unemployment", "employment change", "nonfarm", "non farm",
          "jobless", "wage", "labor force"), "labor"),
        (("pmi",), "pmi"),
        (("interest rate", "rate decision", "fed funds", "bank rate"), "rate"),
        (("retail", "consumer spend", "redbook"), "consumption"),
        (("trade", "current account", "balance", "imports", "exports"), "trade"),
        (("housing", "building", "construction", "house price", "mortgage"), "housing"),
        (("manufacturing", "industrial", "factory", "producer price"), "manufacturing"),
        (("confidence", "sentiment", "optimism", "leading economic"), "sentiment"),
        (("money supply", "credit", "lending", "loan", "mortgage rate"), "monetary"),
        (("oil", "commodity", "gold", "gasoline", "natural gas"), "commodity"),
        (("bond", "bill yield", "treasury", "yield"), "bonds"),
        (("foreign", "investment", "capital flow"), "flows"),
    ]
    for keys, label in rules:
        if any(k in ind for k in keys):
            return label
    if ind == "calendar":
        return "drop"
    return "other"


def process_calendar(df_calander: pd.DataFrame, pairs=PAIRS) -> pd.DataFrame:
    df_cal = df_calander.copy()
    df_cal = df_cal[df_cal["country"].isin(COUNTRY_TO_CURRENCY.keys())].copy()
    df_cal["currency"] = df_cal["country"].map(COUNTRY_TO_CURRENCY)

    df_cal["indicator"] = df_cal["indicator"].fillna("").astype(str)
    df_cal["event_type"] = df_cal["indicator"].apply(categorize_event)
    df_cal = df_cal[df_cal["event_type"] != "drop"].copy()

    # WHY (v3): the old map {-1:1, 0:2, 1:3} barely separated a rate decision
    # from a filler release, and impacts are SUMMED per day. In your file only
    # 4,667 of 189,835 rows are importance=1 while 130,565 are importance=-1,
    # so six trivial prints (6 x 1) outweighed one CPI (1 x 3) and macro_k
    # ended up measuring event COUNT rather than event IMPORTANCE.
    # NEW: a spread that makes one high-importance release worth ~16 low ones.
    # Tune these three numbers if the ablation says the calendar is too weak.
    importance_map = {-1: 0.25, 0: 1.0, 1: 4.0}
    df_cal["impact_score"] = df_cal["importance"].map(importance_map).fillna(0.25)

    df_cal["actual"] = pd.to_numeric(df_cal["actual"], errors="coerce")
    df_cal["forecast"] = pd.to_numeric(df_cal["forecast"], errors="coerce")
    df_cal["surprise"] = df_cal["actual"] - df_cal["forecast"]
    df_cal["surprise_norm"] = (
        df_cal["surprise"] / (df_cal["forecast"].abs() + 1e-6)
    ).clip(-10, 10)

    if "pair" in df_cal.columns:
        df_cal = df_cal.drop(columns=["pair"])
    df_cal["pairs"] = df_cal["currency"].apply(
        lambda c: [] if pd.isna(c) else [p for p in pairs if c in p]
    )
    df_cal = df_cal.explode("pairs").dropna(subset=["pairs"]).rename(columns={"pairs": "pair"})

    df_cal["is_base"] = df_cal.apply(
        lambda row: str(row["pair"]).startswith(str(row["currency"])), axis=1
    )
    df_cal["direction"] = df_cal["is_base"].map({True: 1, False: -1})
    df_cal["directional_impact"] = df_cal["impact_score"] * df_cal["direction"]
    df_cal["surprise_directional"] = df_cal["surprise_norm"] * df_cal["direction"]

    # WHY: the source timestamps are Europe/London. tz_convert(None) moves them
    # to UTC first, so during BST an event before 01:00 local would land on the
    # previous calendar day. Normalize in LOCAL time, then drop the tz, so the
    # release keeps the date it was actually published on.
    df_cal["Date"] = pd.to_datetime(df_cal["date"], errors="coerce")
    if df_cal["Date"].dt.tz is not None:
        df_cal["Date"] = df_cal["Date"].dt.tz_localize(None)   # OLD: .dt.tz_convert(None)
    df_cal["Date"] = df_cal["Date"].dt.normalize()

    return df_cal


def build_pair_features(df_cal: pd.DataFrame) -> pd.DataFrame:
    pair_daily = df_cal.groupby(["Date", "pair"]).agg(
        directional_impact=("directional_impact", "sum"),
        surprise_norm=("surprise_directional", "mean"),
        impact_score=("impact_score", "sum"),
    ).reset_index()

    pair_daily_type = df_cal.groupby(["Date", "pair", "event_type"]).agg(
        directional_impact=("directional_impact", "sum"),
        surprise_norm=("surprise_directional", "mean"),
    ).reset_index()

    directional_piv = pair_daily.pivot(index="Date", columns="pair", values="directional_impact")
    directional_piv.columns = [f"macro_{c}" for c in directional_piv.columns]

    surprise_piv = pair_daily.pivot(index="Date", columns="pair", values="surprise_norm")
    surprise_piv.columns = [f"macro_surprise_{c}" for c in surprise_piv.columns]

    hi = pair_daily_type[pair_daily_type["event_type"].isin(HIGH_IMPACT_TYPES)].copy()
    type_piv = hi.pivot_table(index="Date", columns=["pair", "event_type"],
                              values="directional_impact", aggfunc="sum")
    type_piv.columns = [f"macro_{p}_{t}" for p, t in type_piv.columns]

    pair_features = pd.concat([directional_piv, surprise_piv, type_piv], axis=1)

    # WHY: the rolling below used to run over the PIVOT'S OWN ROWS, which are
    # only the dates that happened to contain an event. "3 rows back" could
    # therefore span a week or more, and did not line up with the panel's
    # trading days at all. Reindexing onto a continuous daily calendar first
    # makes `_3d` mean a real trailing 3 calendar days.
    full_idx = pd.date_range(pair_features.index.min(), pair_features.index.max(), freq="D")
    pair_features = pair_features.reindex(full_idx).fillna(0).rename_axis("Date")
    pair_features = pair_features.sort_index().reset_index()

    cal_cols = [c for c in pair_features.columns if c.startswith("macro_")]
    roll_cols = [c for c in cal_cols if not any(t in c for t in HIGH_IMPACT_TYPES)]

    rolling = {}
    for col in roll_cols:
        rolling[f"{col}_3d"] = pair_features[col].rolling(3, min_periods=1).sum()

        # WHY (LEAKAGE FIX): the old code gave a shift(-1) "known future" lead
        # to EVERY column in roll_cols, which included macro_surprise_<pair>.
        # A scheduled release really is known in advance, so a lead on the
        # directional-impact column is legitimate. A SURPRISE is
        # actual - forecast: it does not exist until the number is published,
        # so macro_surprise_<pair>_lead1 handed the model tomorrow's answer.
        # OLD: rolling[f"{col}_lead1"] = pair_features[col].shift(-1)
        # NEW: leads only for the scheduled-impact columns.
        if not col.startswith("macro_surprise_"):
            rolling[f"{col}_lead1"] = pair_features[col].shift(-1)

    pair_features = pd.concat([pair_features, pd.DataFrame(rolling)], axis=1).fillna(0)
    return pair_features


def _macro_field_template(macro_wide: pd.DataFrame, pairs) -> list:
    """
    Every macro column is named after a pair (macro_AUDCAD=X_rate). Replace the
    pair with a placeholder to get the canonical FIELD list, so that a pair
    missing one event type still yields the same columns (filled with 0)
    instead of changing the schema for that group.
    """
    tmpl = set()
    for p in pairs:
        for c in macro_wide.columns:
            if p in c:
                tmpl.add(c.replace(p, "{S}"))
    return sorted(tmpl)


def merge_calendar_per_slot(panel: pd.DataFrame, pair_features: pd.DataFrame) -> pd.DataFrame:
    """
    WHY:
      OLD: merge_calendar_per_group() selected the right columns per group,
           but named them after the pair. Groups therefore had different
           column sets, and
               dcc_garch_calendar_df = pd.concat(out_rows, ignore_index=True)
               ...fillna(0)
           took the union of all 277 macro columns and filled every
           not-in-this-group cell with a hard 0. The per-group selection was
           correct; the concat that followed undid it.

      NEW: identical selection, but renamed to macro_1 / macro_surprise_2 /
           macro_3_inflation ... so all groups share one schema. The concat
           has no gaps, and the remaining fillna(0) only means "no scheduled
           event for this pair on this day", which is the true meaning.
    """
    macro_wide = pair_features.set_index("Date")
    all_pairs = sorted({p for k in (1, 2, 3) for p in panel[f"pair_{k}"].unique()})
    template = _macro_field_template(macro_wide, all_pairs)

    frames = []
    for gid, g in panel.groupby("group_id", sort=False):
        pairs_in_group = [g[f"pair_{k}"].iloc[0] for k in (1, 2, 3)]
        merged = g

        for k, pair in enumerate(pairs_in_group, start=1):
            wanted = [t.replace("{S}", pair) for t in template]
            sub = macro_wide.reindex(columns=wanted).fillna(0.0)
            sub.columns = [t.replace("{S}", str(k)) for t in template]
            merged = merged.merge(sub.rename_axis("Date").reset_index(),
                                  on="Date", how="left")
        frames.append(merged)

    out = pd.concat(frames, ignore_index=True)
    macro_cols = [c for c in out.columns if c.startswith("macro_")]
    out[macro_cols] = out[macro_cols].fillna(0)
    return out


In [ ]:
# ---------------------------------------------------------------------------
# 10. GDELT MERGE  (scale-free fields only)
# ---------------------------------------------------------------------------
# WHAT THIS DATA ACTUALLY IS -- read this before interpreting any GDELT column.
# The BigQuery export filters `GoldsteinScale <= -5.0`, i.e. CONFLICTUAL events
# only. So avg_tone here is the tone of conflict coverage, not general news
# sentiment, and mentions/articles are conflict-story volume. Treat the whole
# block as a GEOPOLITICAL STRESS measure, not a sentiment measure.
#
# WHY the field list changed:
#   1. mentions_diff / articles_diff / shock_events_diff are COUNT differences,
#      dominated by how much press a country gets rather than by news. Real
#      rows for AUDUSD=X: mentions_diff = -16766, -15887, -14853 every day --
#      that is the US out-publishing Australia ~10x, not information.
#      Worse, GDELT's own sourcing expanded sharply in 2015: mean per-row
#      mentions went 4,429 (2010) -> 23,069 (2015) -> 12,168 (2024). That is a
#      5x level break sitting in the middle of the training window.
#      FIX: counts become a RELATIVE SHARE, (base-quote)/(base+quote), bounded
#      in [-1,1]. Both sides inflate together, so the break divides out.
#   2. articles_diff dropped: corr(mentions_diff, articles_diff) = 1.000 exactly.
#   3. avg_goldstein_diff KEPT. An earlier draft dropped it as "moves with
#      tone" -- that was wrong: corr(avg_goldstein_diff, avg_tone_diff) = 0.004.
#      It is narrow (the filter clamps it to [-10,-5], std 0.47) but it is
#      INDEPENDENT information: how severe the severe events were.
#   4. conflict_share_diff / tone_all_diff are built ONLY if the newer BigQuery
#      export is used (the one that keeps total_event_count / total_mentions /
#      avg_tone_all alongside the filtered figures). conflict_share =
#      shocks / all events is immune to the 2015 coverage break by
#      construction. If those columns are absent the pipeline runs without
#      them and says so.

GDELT_BASE_COLS = ["avg_tone_diff", "avg_goldstein_diff", "shock_rel", "mentions_rel"]
GDELT_RAW_COLS = list(GDELT_BASE_COLS)          # extended by process_gdelt()
GDELT_ROLL_COLS = ["avg_tone_diff", "shock_rel"]  # rolling only where it earns its place


def _relative_share(base: pd.Series, quote: pd.Series) -> pd.Series:
    """(base - quote) / (|base| + |quote|): bounded [-1, 1], size-neutral."""
    return (base - quote) / (base.abs() + quote.abs() + 1.0)


def process_gdelt(gdelt_df: pd.DataFrame) -> pd.DataFrame:
    global GDELT_RAW_COLS, GDELT_ROLL_COLS
    gdelt_df = gdelt_df.copy()

    gdelt_df["Date"] = pd.to_datetime(gdelt_df["d"], errors="coerce")
    if gdelt_df["Date"].dt.tz is not None:
        gdelt_df["Date"] = gdelt_df["Date"].dt.tz_localize(None)
    gdelt_df["Date"] = gdelt_df["Date"].dt.normalize()
    gdelt_df = gdelt_df.rename(columns={"ticker": "pair"})

    # the AVG columns were never COALESCEd in the query, but the NULL rate is
    # only 0.11% / 0.21%, so a plain zero-fill is harmless here.
    for c in ["base_avg_tone", "quote_avg_tone", "avg_tone_diff",
              "base_avg_goldstein", "quote_avg_goldstein", "avg_goldstein_diff"]:
        if c in gdelt_df.columns:
            gdelt_df[c] = gdelt_df[c].fillna(0)

    # scale-free versions, built from the base/quote levels the old pipeline
    # read out of the file and then never used
    gdelt_df["shock_rel"] = _relative_share(gdelt_df["base_shock_events"],
                                            gdelt_df["quote_shock_events"])
    gdelt_df["mentions_rel"] = _relative_share(gdelt_df["base_mentions"],
                                               gdelt_df["quote_mentions"])

    GDELT_RAW_COLS = list(GDELT_BASE_COLS)
    GDELT_ROLL_COLS = ["avg_tone_diff", "shock_rel"]

    # --- optional, only present with the newer unfiltered-baseline query ----
    have_totals = {"base_total_event_count", "quote_total_event_count"} <= set(gdelt_df.columns)
    if have_totals:
        b = gdelt_df["base_shock_events"] / gdelt_df["base_total_event_count"].replace(0, np.nan)
        q = gdelt_df["quote_shock_events"] / gdelt_df["quote_total_event_count"].replace(0, np.nan)
        gdelt_df["conflict_share_diff"] = (b - q).fillna(0)
        GDELT_RAW_COLS.append("conflict_share_diff")
        GDELT_ROLL_COLS.append("conflict_share_diff")
        print("  GDELT: conflict_share_diff built (unfiltered baseline present)")
    else:
        print("  GDELT: no total_event_count columns -- conflict_share_diff skipped. "
              "Re-run the BigQuery export with conditional aggregation to enable it.")

    if {"base_avg_tone_all", "quote_avg_tone_all"} <= set(gdelt_df.columns):
        gdelt_df["tone_all_diff"] = (gdelt_df["base_avg_tone_all"].fillna(0)
                                     - gdelt_df["quote_avg_tone_all"].fillna(0))
        GDELT_RAW_COLS.append("tone_all_diff")
        print("  GDELT: tone_all_diff built (general-news tone available)")

    gdelt_df[GDELT_RAW_COLS] = gdelt_df[GDELT_RAW_COLS].fillna(0)
    return gdelt_df


def merge_gdelt_per_slot(panel: pd.DataFrame, gdelt_df: pd.DataFrame) -> pd.DataFrame:
    """
    WHY:
      OLD merge_gdelt_per_group() named columns after the pair, so groups had
      different column sets and the closing
          result = pd.concat(out_rows, ignore_index=True); result.fillna(0)
      rebuilt the full-width union and turned "this pair is not in this group"
      into a hard zero -- the 75-95%-zeros pattern in the old CSV.
      NEW: columns named by SLOT, so every group emits one identical schema,
      the concat has no gaps, and the remaining fillna(0) only ever means
      "GDELT reported nothing that day".
    """
    gd = gdelt_df.set_index(["Date", "pair"])[GDELT_RAW_COLS].sort_index()
    known_pairs = set(gdelt_df["pair"].unique())
    frames = []

    for gid, g in panel.groupby("group_id", sort=False):
        pairs_in_group = [g[f"pair_{k}"].iloc[0] for k in (1, 2, 3)]
        merged = g
        for k, pair in enumerate(pairs_in_group, start=1):
            slot_names = {c: f"{c}_{k}" for c in GDELT_RAW_COLS}
            if pair in known_pairs:
                sub = gd.xs(pair, level="pair").rename(columns=slot_names)
                sub = sub[~sub.index.duplicated(keep="first")]
            else:
                print(f"WARNING: pair {pair} (group {gid}) not found in GDELT data")
                sub = pd.DataFrame(0.0,
                                   index=pd.Index(g["Date"].unique(), name="Date"),
                                   columns=list(slot_names.values()))
            merged = merged.merge(sub.rename_axis("Date").reset_index(),
                                  on="Date", how="left")
        frames.append(merged)

    result = pd.concat(frames, ignore_index=True)
    slot_cols = [f"{c}_{k}" for c in GDELT_RAW_COLS for k in (1, 2, 3)]
    result[slot_cols] = result[slot_cols].fillna(0)      # fill BEFORE averaging
    for c in GDELT_RAW_COLS:
        result[c] = result[[f"{c}_{k}" for k in (1, 2, 3)]].mean(axis=1)
    return result


In [ ]:
# ---------------------------------------------------------------------------
# 11. REGIMES (expanding-window cutoffs -- fixes leakage)
# ---------------------------------------------------------------------------
def _expanding_percentile_regime(x: pd.Series) -> pd.Series:
    """3-state regime using EXPANDING 33rd/67th percentiles."""
    low = x.expanding(min_periods=30).quantile(0.33)
    high = x.expanding(min_periods=30).quantile(0.67)
    low, high = low.bfill(), high.bfill()
    regime = pd.Series(1, index=x.index)
    regime[x <= low] = 0
    regime[x >= high] = 2
    return regime.astype(int)


def add_regimes(df: pd.DataFrame) -> pd.DataFrame:
    """
    WHY:
      OLD: add_regimes(df, vol_col="sigma_AUDUSD=X") -- the volatility regime
      for EVERY group was computed from AUDUSD's volatility, even for groups
      containing no AUD and no USD pair. Only possible because sigma was
      broadcast to all 27 pairs. NEW: uses the mean sigma of this group's own
      three legs.
      Also v3: only ONE interaction is kept. tone_mentions_interaction moved
      almost identically to tone_shock_interaction, so it was pure duplication.
    """
    dff = df.copy()
    dff["regime"] = 0
    dff["tone_shock_interaction"] = dff["avg_tone_diff"] * dff["shock_rel"]

    # OLD: dff.groupby("group_id")["sigma_AUDUSD=X"].transform(...)
    dff["sigma_group"] = dff[["sigma_1", "sigma_2", "sigma_3"]].mean(axis=1)
    dff["regime_vol"] = dff.groupby("group_id")["sigma_group"].transform(_expanding_percentile_regime)

    # macro_pressure now counts only this group's own macro columns.
    macro_cols = [c for c in dff.columns if c.startswith("macro_")]
    if macro_cols:
        dff["macro_pressure"] = (dff[macro_cols].abs() > 3).sum(axis=1)
        dff["regime_macro"] = dff.groupby("group_id")["macro_pressure"].transform(
            _expanding_percentile_regime)
    else:
        dff["macro_pressure"] = 0
        dff["regime_macro"] = 0

    # HMM news regime -- fit ONCE per group on the full history. Like any
    # global HMM fit this still uses full-sample info for the PARAMETERS, so
    # EXCLUDE `regime` from the feature list for an honest train/test split,
    # or refit it per fold. regime_vol / regime_macro are expanding and fine.
    for gid, group in dff.groupby("group_id"):
        if len(group) < 50:
            continue
        cols = [c for c in list(GDELT_RAW_COLS) + ["tone_shock_interaction"]
                if c in group.columns]
        cols += [f"{f}_{k}" for f in GDELT_RAW_COLS for k in (1, 2, 3)
                 if f"{f}_{k}" in group.columns]
        if len(cols) < 2:
            continue
        X = StandardScaler().fit_transform(group[cols].fillna(0).values)
        model = GaussianHMM(n_components=3, covariance_type="diag",
                            n_iter=1000, tol=1e-4, min_covar=1e-2, random_state=42)
        try:
            model.fit(X)
            dff.loc[group.index, "regime"] = model.predict(X)
        except Exception as e:
            print(f"HMM failed for {gid}: {e}")
    return dff


In [ ]:
# ---------------------------------------------------------------------------
# 12. ROLLING GDELT FEATURES + EVENT WINDOW (no future GDELT leakage)
# ---------------------------------------------------------------------------
# WHY: the original built mean+std at 3, 7 AND 14 days plus diff_1 and diff_3
# for five fields -- 40 columns, most of them near-copies (a 3-day and a 7-day
# mean of the same series correlate ~0.9). One window (7) over the two or three
# fields that carry the signal gives the same information in 6-9 columns.
ROLL_WINDOW = 7


def add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    gdelt_cols = [c for c in GDELT_ROLL_COLS if c in df.columns]

    new_features = []
    for col in gdelt_cols:
        grp = df.groupby("group_id")[col]
        new_features.append(grp.transform(lambda x: x.rolling(ROLL_WINDOW, min_periods=1).mean())
                            .rename(f"{col}_mean_{ROLL_WINDOW}"))
        new_features.append(grp.transform(lambda x: x.rolling(ROLL_WINDOW, min_periods=2).std())
                            .rename(f"{col}_std_{ROLL_WINDOW}"))
        new_features.append(grp.diff(1).rename(f"{col}_diff_1"))
    df = pd.concat([df] + new_features, axis=1)

    if "shock_rel" in df.columns:
        # WHY event_flag fired on 48% of days instead of 20%:
        #   OLD: x.abs() > x.expanding(min_periods=30).quantile(0.80).bfill()
        #        left side |x|, right side the 80th percentile of x ITSELF, so
        #        it fired when x > q80 (20%) OR x < -q80 (another 20%) -- and
        #        more once the series skewed. The .bfill() also back-filled the
        #        warm-up rows with a threshold built from FUTURE data.
        #   NEW: |x| against the expanding 80th percentile OF |x|; warm-up rows
        #        stay unflagged (NaN comparison is False) instead of borrowing
        #        a future cutoff.
        df["event_flag"] = df.groupby("group_id")["shock_rel"].transform(
            lambda x: (x.abs() > x.abs().expanding(min_periods=30).quantile(0.80))
        ).astype(int)
        df["event_lag_1"] = df.groupby("group_id")["event_flag"].shift(1).fillna(0)

    protect_cols = ["regime", "regime_vol", "regime_macro", "group_id", "Date",
                    "group", "pair_1", "pair_2", "pair_3"]
    fill_cols = [c for c in df.columns if c not in protect_cols]
    df[fill_cols] = df[fill_cols].fillna(0)
    return df.copy()


In [ ]:
# ---------------------------------------------------------------------------
# 13. FINAL FEATURE SELECTION
# ---------------------------------------------------------------------------
# WHY: the pipeline still BUILDS columns we do not want to train on (the minor
# macro event types, the _3d windows, the raw spread). Rather than thread the
# trimming through every function -- which makes each one harder to read and
# easier to break -- everything is generated as before and this one whitelist
# decides what reaches the CSV. To add a feature back, add its name here.
#
# ROLE ASSIGNMENTS for NeuralForecast:
#   unique_id   = group_id
#   ds          = Date
#   y           = zscore.groupby(group_id).shift(-1) - zscore   (built downstream)
#   static_exog = pair_1, pair_2, pair_3
#   futr_exog   = FUTR_EXOG below -- scheduled releases and the weekday are
#                 genuinely known in advance. This is the ONLY legitimate
#                 look-ahead in the dataset.
#   hist_exog   = everything else
#
# NOTE: exclude `regime` from hist_exog for an honest train/test split -- the
# HMM behind it is fit once over the full history. regime_vol and regime_macro
# use expanding windows and are safe.

ID_COLS = ["Date", "group", "group_id", "pair_1", "pair_2", "pair_3", "time_idx"]
TARGET_COLS = ["zscore", "spread"]

PRICE_FEATURES = (
    [f"ret_{k}" for k in (1, 2, 3)]
    + [f"ret_{k}_lag1" for k in (1, 2, 3)]
    + [f"ret_{k}_mean_5" for k in (1, 2, 3)]
    + [f"ret_{k}_std_20" for k in (1, 2, 3)]
    + [f"sigma_{k}" for k in (1, 2, 3)]
    + ["rho_12", "rho_13", "rho_23",
       "rho_12_change", "rho_13_change", "rho_23_change",
       "sigma_group", "zscore_lag1", "zscore_diff_1", "zscore_diff_5"]
)

# Kept: overall directional impact, the surprise, and the three event types
# that actually move FX. Dropped: growth / pmi / trade types and the _3d windows.
CALENDAR_FEATURES = (
    [f"macro_{k}" for k in (1, 2, 3)]
    + [f"macro_surprise_{k}" for k in (1, 2, 3)]
    + [f"macro_{k}_rate" for k in (1, 2, 3)]
    + [f"macro_{k}_inflation" for k in (1, 2, 3)]
    + [f"macro_{k}_labor" for k in (1, 2, 3)]
    + ["macro_pressure", "regime_macro"]
)

FUTR_EXOG = [f"macro_{k}_lead1" for k in (1, 2, 3)] + ["dow"]

OTHER_FEATURES = ["regime_vol"]


def build_final_columns() -> list:
    """Built at call time so the optional GDELT fields are picked up."""
    gdelt = (
        [f"{f}_{k}" for f in GDELT_RAW_COLS for k in (1, 2, 3)]      # per slot
        + list(GDELT_RAW_COLS)                                       # group aggregate
        + [f"{f}_mean_{ROLL_WINDOW}" for f in GDELT_ROLL_COLS]
        + [f"{f}_std_{ROLL_WINDOW}" for f in GDELT_ROLL_COLS]
        + [f"{f}_diff_1" for f in GDELT_ROLL_COLS]
        + ["tone_shock_interaction", "event_flag", "event_lag_1", "regime"]
    )
    return (ID_COLS + TARGET_COLS + PRICE_FEATURES
            + CALENDAR_FEATURES + FUTR_EXOG + gdelt + OTHER_FEATURES)


def select_final_columns(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    final_columns = build_final_columns()
    keep = [c for c in final_columns if c in df.columns]
    missing = [c for c in final_columns if c not in df.columns]
    dropped = [c for c in df.columns if c not in final_columns]
    if verbose:
        print(f"  keeping {len(keep)} columns, dropping {len(dropped)}")
        if missing:
            print(f"  WARNING -- expected but not built: {missing}")
    return df[keep].copy()


In [ ]:
# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main(economic_data_path="/content/drive/MyDrive/economic_data.feather",
         gdelt_path="/content/drive/MyDrive/gdelt.csv",
         # NEW FILENAME ON PURPOSE -- the schema changed, so this must not
         # silently overwrite the CSV the current training notebook reads.
         output_path="dcc_garch_calander_gdelt_final_v4.csv"):
    print("1/9  Downloading prices...")
    data = download_prices()
    returns = data.pct_change(fill_method=None).dropna()

    print("2/9  Ranking correlated pairs...")
    top_pairs = top_correlated_pairs(returns)
    corr_dict = build_corr_dict(top_pairs)

    print("3/9  Building cointegrated groups (Johansen)...")
    groups_90, groups_95 = build_group_sets(corr_dict, data)
    groups = groups_95

    print("4/9  Fitting VECM, extracting residuals + spread...")
    vecm_models, vecm_residuals, spread_dict, zscore_dict = fit_vecm_for_groups(data, groups)

    print("5/9  Fitting GARCH (expanding-window standardized)...")
    garch_models, garch_volatility, garch_std_resid = fit_garch_on_residuals(vecm_residuals)
    returns_volatility = fit_garch_on_returns(returns)

    print("6/9  Fitting DCC, building panel (slot schema + price features)...")
    final_dcc_df = build_dcc_dataset(garch_std_resid)
    panel = build_panel(final_dcc_df, returns_volatility, spread_dict,
                        zscore_dict, returns=returns)      # returns now passed in

    print("7/9  Merging macro calendar (per slot)...")
    df_calander = (pd.read_feather(economic_data_path)
                   if economic_data_path.endswith(".feather")
                   else pd.read_csv(economic_data_path))
    df_cal = process_calendar(df_calander)
    pair_features = build_pair_features(df_cal)
    panel = merge_calendar_per_slot(panel, pair_features)

    print("8/9  Merging GDELT (per slot), building regimes...")
    gdelt_df = process_gdelt(pd.read_csv(gdelt_path))
    panel = merge_gdelt_per_slot(panel, gdelt_df)
    panel = add_regimes(panel)

    print("9/9  Rolling features, selecting final columns, exporting...")
    final_df = add_rolling_features(panel)
    final_df = select_final_columns(final_df)
    final_df = final_df.sort_values(["group_id", "Date"]).reset_index(drop=True)

    final_df.to_csv(output_path, index=False)
    print(f"Done. Shape: {final_df.shape}. Saved to {output_path}")
    return final_df


if __name__ == "__main__":
    main(
        economic_data_path="/content/drive/MyDrive/economic_data.feather",
        gdelt_path="/content/drive/MyDrive/gdelt.csv",
        output_path="/content/drive/MyDrive/dcc_garch_calander_gdelt_final_v4.csv",
    )


Check Dataset


In [ ]:
# ---------------------------------------------------------------------------
# CHECK DATASET -- the checks that would have caught the original bugs
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np

df = pd.read_csv("/content/drive/MyDrive/dcc_garch_calander_gdelt_final_v4.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["group_id", "Date"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Groups:", df.group_id.nunique(), "| Days:", df.Date.nunique())
print("Rows per group:", df.groupby("group_id").size().unique())
print("Duplicate (Date, group_id):", df.duplicated(["Date", "group_id"]).sum())
print("NaNs:", int(df.isna().sum().sum()))

ID = ["group", "group_id", "pair_1", "pair_2", "pair_3"]

# [1] THE CHECK THAT MATTERS: constant columns PER GROUP, not globally. The old
#     file passed a global check with "0 dead columns" while ~575 of its 616
#     columns were frozen constants inside every individual group.
const = (df.groupby("group_id").nunique() <= 1).sum(axis=0)
print("\n[1] constant in 2+ groups (should be empty):",
      [c for c in const[const >= 2].index if c not in ID])

# [2] every group must have the identical schema
print("[2] populated column count per group (should be one value):",
      set(int(s.notna().all().sum()) for _, s in df.groupby("group_id")))

# [3] no forward-looking surprise columns
print("[3] leaky surprise leads (should be empty):",
      [c for c in df.columns if c.startswith("macro_surprise") and c.endswith("lead1")])

# [4] event_flag ~20%, not ~48%
print("[4] event_flag rate: %.3f" % df.event_flag.mean())

# [5] GDELT counts must be scale-free, not dominated by country size
for c in [c for c in ["shock_rel", "mentions_rel", "conflict_share_diff"] if c in df.columns]:
    print(f"[5] {c:20s} min {df[c].min():+.3f}  max {df[c].max():+.3f}  (bounded)")

# [6] rho changes should almost never be exactly zero (was 90-95%)
for c in ["rho_12_change", "rho_13_change", "rho_23_change"]:
    print(f"[6] {c}: zero fraction = {(df[c] == 0).mean():.4f}")

# [7] no 2015 coverage break left in the GDELT features
if "mentions_rel" in df.columns:
    yearly = df.groupby(df.Date.dt.year)["mentions_rel"].mean()
    print("[7] mentions_rel yearly mean spread: %.3f (a big number means the "
          "coverage break survived)" % (yearly.max() - yearly.min()))

# [8] TARGET + the baselines every model must beat
target = df.groupby("group_id")["zscore"].shift(-1) - df["zscore"]
naive = df.groupby("group_id")["zscore"].diff(1)
print("\n[8] target  mean %.4f  std %.4f" % (target.mean(), target.std()))
print("    baseline MAE, predict 0     : %.4f" % target.abs().mean())
print("    baseline MAE, predict naive : %.4f" % (target - naive).abs().mean())

# [9] feature balance by source -- price must not be a rounding error
import collections
fam = collections.Counter()
for c in df.columns:
    if   c.startswith(("ret_", "sigma", "rho_", "zscore_")):            fam["price"] += 1
    elif c.startswith("macro_") or c in ("regime_macro", "dow"):        fam["calendar"] += 1
    elif c.startswith(("avg_", "shock_", "mentions_", "tone_", "event_",
                       "conflict_")) or c == "regime":                  fam["gdelt"] += 1
print("\n[9] feature counts by source:", dict(fam))
